In [9]:
import os
import cv2
import numpy as np
from glob import glob
from tqdm import tqdm
import zipfile

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms

import segmentation_models_pytorch as smp

In [10]:
train_images = "train/images"
train_masks = "train/masks"
test_images = "test/images"

class GlitchDataset(Dataset):
    def __init__(self, image_dir, mask_dir=None, transform=None):
        self.image_paths = sorted(glob(os.path.join(image_dir, "*.png")))
        self.mask_paths = sorted(glob(os.path.join(mask_dir, "*.png"))) if mask_dir else None
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img = cv2.imread(self.image_paths[idx])
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (256, 256))
        
        if self.mask_paths:
            mask = cv2.imread(self.mask_paths[idx], cv2.IMREAD_GRAYSCALE)
            mask = cv2.resize(mask, (256, 256))
            mask = mask / 255.0
            mask = np.expand_dims(mask, axis=0)
        else:
            mask = None

        img = img / 255.0
        img = np.transpose(img, (2, 0, 1))

        if mask is not None:
            return torch.tensor(img, dtype=torch.float32), torch.tensor(mask, dtype=torch.float32)
        else:
            return torch.tensor(img, dtype=torch.float32), self.image_paths[idx]

train_dataset = GlitchDataset(train_images, train_masks)
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, num_workers=0)

test_dataset = GlitchDataset(test_images)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False, num_workers=0)

In [14]:
epochs = 17
lr = 0.002

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = smp.Unet(
    encoder_name="resnet34",
    encoder_weights="imagenet",
    in_channels=3,
    classes=1,
).to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=lr)

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    for imgs, masks in tqdm(train_loader):
        imgs, masks = imgs.to(device), masks.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, masks)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        running_loss += loss.item()
    print(f"Epoch {epoch+1}/{epochs}, Loss: {running_loss/len(train_loader):.4f}")

100%|██████████████████████████████████████████████████████████████████████████████████| 50/50 [00:06<00:00,  8.14it/s]


Epoch 1/17, Loss: 0.1994


100%|██████████████████████████████████████████████████████████████████████████████████| 50/50 [00:06<00:00,  8.02it/s]


Epoch 2/17, Loss: 0.0633


100%|██████████████████████████████████████████████████████████████████████████████████| 50/50 [00:06<00:00,  7.73it/s]


...

Epoch 15/17, Loss: 0.0129


100%|██████████████████████████████████████████████████████████████████████████████████| 50/50 [00:05<00:00,  9.13it/s]


Epoch 16/17, Loss: 0.0081


100%|██████████████████████████████████████████████████████████████████████████████████| 50/50 [00:05<00:00,  8.96it/s]

Epoch 17/17, Loss: 0.0073


In [15]:
output_dir = "pred_masks"

os.makedirs(output_dir, exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.eval()
with torch.no_grad():
    for imgs, paths in tqdm(test_loader):
        imgs = imgs.to(device)
        outputs = model(imgs)
        outputs = torch.sigmoid(outputs)
        outputs = (outputs > 0.5).float()
        outputs = outputs.cpu().numpy()
        for out, path in zip(outputs, paths):
            mask = (out[0]*255).astype(np.uint8)
            filename = os.path.basename(path)
            cv2.imwrite(os.path.join(output_dir, filename), mask)

100%|████████████████████████████████████████████████████████████████████████████████| 200/200 [00:04<00:00, 45.80it/s]


In [16]:
def zip_masks(mask_dir, zip_name):
    with zipfile.ZipFile(zip_name, 'w') as zipf:
        for file in os.listdir(mask_dir):
            zipf.write(os.path.join(mask_dir, file), arcname=file)

zip_masks("pred_masks", zip_name="submission.zip")